# MA stock surrogate: download and run

The MA surrogate is ten LightGBM models, one per output: `Energy.*` and `Peak.*` for
Heating, Cooling, Domestic Hot Water, Equipment and Lighting. Each takes the same
194 features described by `space.yml`. The model files stay in S3. This repo
records which files they are (by sha256) in
`outputs/surrogates/MA/<version>/surrogate-version.yml`.

**Run it:**
```bash
uv sync --extra surrogate
uv run --extra surrogate --with jupyterlab jupyter lab examples/ma-surrogate.ipynb
```

**Units:** `Energy.*` is kWh/m²/yr and `Peak.*` is kW/m², per conditioned floor area.
Both are **demand before system efficiency**: switching a gas furnace to a heat pump
does not change `Energy.Heating`. Apply COPs yourself to get site energy.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

import lightgbm as lgb
import numpy as np
import pandas as pd
import yaml

from globi_data import paths, surrogates

ROOT = paths.repo_root()
VERSION = "current"  # pin e.g. "v001" for reproducible results

sdir = paths.surrogate_dir(ROOT, "MA")
version = paths.resolve_version(sdir, VERSION)
manifest = surrogates.read_surrogate_manifest(sdir / version)
print(f"MA {version}: {manifest.summary}")
print(f"training run: {manifest.provenance.training.experiment_id}")
pd.DataFrame([t.model_dump() for t in manifest.targets])

## 1. Download (once)

`fetch_surrogate` downloads into `~/.cache/globi-data/surrogates/MA/<version>/` and checks
every file against its recorded sha256. Later calls only re-hash, which takes under a second.
If someone overwrites an object in S3, this raises an error rather than handing you a different model.

In [ ]:
model_dir = surrogates.fetch_surrogate(ROOT, "MA", version)
space = yaml.safe_load((model_dir / manifest.transforms.path).read_text())["features"]
FEATURES = [f["name"] for f in space]
sorted(p.name for p in model_dir.iterdir())

## 2. Load the models

Loading runs in a thread pool. LightGBM parses in C and releases the GIL, so the ten files load in parallel.

In [ ]:
files = sorted(model_dir.glob("*.lgb"))
with ThreadPoolExecutor() as pool:
    boosters = dict(zip(
        (f.stem for f in files),
        pool.map(lambda f: lgb.Booster(model_file=str(f)), files),
    ))
list(boosters)

## 3. Encode features

This is the same transform epengine applies (`RegressorInputSpec.transform`), vectorised and without
the epengine dependency:

- Categorical features become their index in `values`. An unknown value raises an error; epengine would encode it as -1 and still predict.
- Continuous features are min-max scaled to the training range. Out-of-range values only warn, because tree models clamp them silently.

In [ ]:
def encode(df: pd.DataFrame) -> np.ndarray:
    """Raw feature table -> model matrix, columns in space.yml order."""
    cols = []
    for f in space:
        x = df[f["name"]]
        if "values" in f:
            codes = pd.Categorical(x, categories=f["values"]).codes
            if (codes < 0).any():
                bad = sorted(set(x[codes < 0]))
                raise ValueError(f"{f['name']}: {bad} not in {f['values']}")
            cols.append(codes.astype(np.float32))
        else:
            v = x.to_numpy(dtype=np.float64)
            if ((v < f["min"]) | (v > f["max"])).any():
                print(f"warning: {f['name']} outside training range [{f['min']}, {f['max']}]")
            # Features constant in training (min == max) become NaN, as in epengine;
            # LightGBM treats NaN as missing, which is what it saw in training.
            with np.errstate(invalid="ignore", divide="ignore"):
                cols.append(((v - f["min"]) / (f["max"] - f["min"])).astype(np.float32))
    return np.column_stack(cols)


def predict(df: pd.DataFrame) -> pd.DataFrame:
    """Predict every target; columns are (Energy|Peak, end use)."""
    X = encode(df)
    out = pd.DataFrame({name: b.predict(X, num_threads=0) for name, b in boosters.items()}, index=df.index)
    out.columns = pd.MultiIndex.from_tuples([tuple(c.split(".", 1)) for c in out.columns])
    return out

## 4. Inputs

The model needs all 194 `feature.*` columns. In MA-stock-analysis they come from the GIS
pipeline: geometry and shading from the footprint and its neighbours, weather statistics from the
nearest EPW, and semantic fields from priors.

The cell below builds a single building by hand. Columns you don't set get the first category, or
the midpoint of the training range. That is enough to try the model, but the result is not calibrated:
the model is very sensitive to window-to-wall ratio and envelope leakiness. Use real geometry,
shading and weather values for real results.

In [ ]:
def building(**overrides) -> dict:
    row = {f["name"]: f["values"][0] if "values" in f else (f["min"] + f["max"]) / 2 for f in space}
    row.update({k: 0.0 for k in row if "shading_mask" in k})  # unshaded
    row.update({f"feature.{k.replace('__', '.')}": v for k, v in overrides.items()})
    return row


house = building(
    geometry__long_edge=10.0, geometry__short_edge=7.5, geometry__aspect_ratio=10 / 7.5,
    geometry__num_floors=2, geometry__energy_model_conditioned_area=150.0,
    geometry__energy_model_occupied_area=150.0, geometry__orientation=0.0,
    geometry__wwr=0.15, geometry__f2f_height=3.0,
    **{"geometry__orientation.cos": 1.0, "geometry__orientation.sin": 0.0},
    semantic__Typology="SFH", semantic__Age_bracket="pre_1975",
    semantic__Heating="NaturalGasHeating", semantic__Cooling="ACCentral",
    semantic__Weatherization="LeakyEnvelope", semantic__Windows="SinglePane",
    weather__file="USA_MA_Boston-Logan.Intl.AP.725090_TMYx.2009-2023",
)
predict(pd.DataFrame([house])).T.rename(columns={0: "per m2"})

## 5. Retrofit delta

Change some semantic fields, predict again, and take the difference. This is how MA-stock-analysis builds its 13 retrofit scenarios.

In [ ]:
before = pd.DataFrame([house])
after = before.assign(**{
    "feature.semantic.Weatherization": "TightEnvelope",
    "feature.semantic.Windows": "DoublePaneLowE",
})
delta = predict(after) - predict(before)
delta["Energy"].T.rename(columns={0: "change, kWh/m2/yr"})

## 6. Many variants in one call

Build every row first, then call `predict` once. Encoding is vectorised, and each booster scores the
whole matrix in one multithreaded call. A Python loop over rows is far slower.

Here that means one house in every combination of walls, windows and weatherization (80 rows).
Whole-stock runs, with millions of buildings × Monte Carlo samples × scenarios and features built from
GIS, don't belong in a notebook. They run as the scythe flow in MA-stock-analysis
(`simulate_ma_stock_building`, one task per census tract), which uses this same encode-once pattern.

In [ ]:
import itertools

options = {f["name"]: f["values"] for f in space if "values" in f}
fields = ["feature.semantic.Walls", "feature.semantic.Windows", "feature.semantic.Weatherization"]
variants = pd.DataFrame(
    [{**house, **dict(zip(fields, combo))} for combo in itertools.product(*(options[f] for f in fields))]
)
t = time.perf_counter()
results = predict(variants)
print(f"{len(variants)} variants in {time.perf_counter() - t:.2f}s")

heating = results[("Energy", "Heating")].set_axis(pd.MultiIndex.from_frame(variants[fields]))
heating.sort_values().head(10)  # lowest heating demand, kWh/m2/yr